In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
# In this notebook I'll standardize data that water keeper alliance sent us
import os
print(os.getcwd())

C:\Users\Nilou Yaar\OneDrive - The Public Interest Network\Desktop\Projects\Large projects\PFAS code


In [3]:
file_path='G:/My Drive/Projects/Large projects/PFAS in water/'

In [4]:
file_path = "G:/My Drive/Projects/Large projects/PFAS in water/"

phase1_path = file_path + "raw_data/WA phase 1/Appendix-4-All-Waterkeeper-Alliance-PFAS Phase I-Sample-Results-Sheet1.xlsx"

phase2_results_path = file_path + "raw_data/WA phase 2/Waterkeeper Alliance Phase II PFAS Results for Environment America - Do Not Share.xlsx"

phase2_locations_path = file_path + "raw_data/WA phase 2/Waterkeeper Alliance PFAS Phase II Location Data for Environment America - Do Not Share.xlsx"

### Cleaning Phase 1 data

In [5]:
# WA sent us two phases and
# Phase I is structured in a wide format where each PFAS analyte
# is stored in its own column. I separate the location/sample
# information from the PFAS result columns so I can reshape the data.

In [6]:
# I load the Phase I Waterkeeper file.
# This file has one row for each sample, but the PFAS results are stored
# across many separate PFAS columns.
phase1 = pd.read_excel(phase1_path)

In [7]:
# I inspect the Phase I file before changing anything.
# I check the first few rows, the size of the dataset and the column names.
display(phase1.head())
print(phase1.shape)
print(phase1.columns.tolist())

,Sample ID,Sampling Date,Waterkeeper Group,State,Waterbody,Upstream or Downstream,Latitude,Longitude,10:2 FTS,11CL-PF3OUdS,...,PFNS,PFOA,PFOS,PFOSA,PFPeA,PFPeS,PFPrS,PFTeA,PFTrDA,PFUnA
0,WKA_2022_0001,2022-06-23,Narragansett Baykeeper,RI,Spring Green Pond,Upstream,41.73500,-71.40810,< 1 ng/L,< 1 ng/L,...,< 1 ng/L,29,34.5,< 1 ng/L,60.7,3.9,1.6,< 1 ng/L,< 1 ng/L,1.6
1,WKA_2022_0002,2022-06-23,Narragansett Baykeeper,RI,Buckeye Brook,Downstream,41.71430,-71.39845,< 1 ng/L,< 1 ng/L,...,< 1 ng/L,7.1,4,< 1 ng/L,3.1,< 1 ng/L,< 1 ng/L,< 1 ng/L,< 1 ng/L,< 1 ng/L
2,WKA_2022_0003,2022-06-28,Upper Missouri Waterkeeper,MT,East Gallatin River,Upstream,45.72502,-111.06720,< 1 ng/L,< 1 ng/L,...,< 1 ng/L,< 1 ng/L,< 1 ng/L,< 1 ng/L,< 1 ng/L,< 1 ng/L,< 1 ng/L,< 1 ng/L,< 1 ng/L,< 1 ng/L
3,WKA_2022_0004,2022-06-28,Upper Missouri Waterkeeper,MT,East Gallatin River,Downstream,45.76852,-111.09783,< 1 ng/L,< 1 ng/L,...,< 1 ng/L,< 1 ng/L,< 1 ng/L,< 1 ng/L,< 1 ng/L,< 1 ng/L,< 1 ng/L,< 1 ng/L,< 1 ng/L,< 1 ng/L
4,WKA_2022_0005,2022-06-20,Humboldt Baykeeper,CA,Eel River,Upstream,40.58018,-124.15648,< 1 ng/L,< 1 ng/L,...,< 1 ng/L,< 1 ng/L,< 1 ng/L,< 1 ng/L,< 1 ng/L,< 1 ng/L,< 1 ng/L,< 1 ng/L,< 1 ng/L,< 1 ng/L


(228, 62)
['Sample ID', 'Sampling Date', 'Waterkeeper Group', 'State', 'Waterbody', 'Upstream or Downstream', 'Latitude', 'Longitude', '10:2 FTS', '11CL-PF3OUdS', '3:3 FTCA', '4:2 FTS', '5:3 FTCA', '6:2 FTS', '6:6PFPi', '6:8PFPi', '7:3 FTCA', '8:2 FTS', '8Cl-PFOS', '9Cl-PF3ONS', 'ADONA', 'FBSA', 'FDSA-I', 'FDUEA', 'FHxSA', 'FOSAA', 'FOUEA', 'HFPO-DA\n(GenX)', 'MeFBSA', 'N-AP-FHxSA', 'N-EtFOSA', 'N-EtFOSAA', 'NEtFOSE', 'NFDHA', 'N-MeFOSA', 'N-MeFOSAA', 'NMeFOSE', 'PFBA', 'PFBS', 'PFDA', 'PFDoA', 'PFDoS', 'PFDS', 'PFECHS', 'PFEESA', 'PFHpA', 'PFHpS', 'PFHxA', 'PFHxS', 'PFMOBA', 'PFMOPrA', 'PFNA', 'PFNS', 'PFOA', 'PFOS', 'PFOSA', 'PFPeA', 'PFPeS', 'PFPrS', 'PFTeA', 'PFTrDA', 'PFUnA']


In [8]:
meta_cols = [
    "Sample ID",
    "Sampling Date",
    "Waterkeeper Group",
    "State",
    "Waterbody",
    "Upstream or Downstream",
    "Latitude",
    "Longitude"
]

pfas_cols = [c for c in phase1.columns if c not in meta_cols]

# I separate the sample and location columns from the PFAS result columns.
# The PFAS result columns need to be reshaped into long format.

In [9]:
# ok I need to Keep the sample information columns exactly as they are, 
#but take all the PFAS columns and stack them underneath each other 
#into one PFAS-name column and one result column
phase1_long = phase1.melt(
    id_vars=meta_cols,
    value_vars=pfas_cols,
    var_name="CharacteristicName",
    value_name="raw_result"
)

# I convert Phase I from wide format (wider table) to long format (more rows)
# Now each row is one PFAS result for one sample,
# which is closer to the structure of the WQP data.

In [10]:
phase1_long["raw_result_text"] = (
    phase1_long["raw_result"]
    .astype("string")
    .str.strip()
)

# I keep a cleaned text version of the original result.
# I need this because values that start with "<" should be kept for now

In [11]:
# I inspect the original Phase I result values before assigning detection status.
# I do this because I do not want to assume that every value starting with "<"
# is the only type of non-detect or that every other value is automatically detected.

phase1_long["raw_result_text"] = (
    phase1_long["raw_result"]
    .astype("string")
    .str.strip()
)

display(
    phase1_long["raw_result_text"]
    .value_counts(dropna=False)
    .head(50)
)

raw_result_text
< 1 ng/L      10131
< 2 ng/L        505
< 10 ng/L       216
< 1.4 ng/L       95
1.1              69
1.3              53
< 0.5 ng/L       51
1.2              51
1.7              49
< 1.1 ng/L       45
< 1.6 ng/L       45
1.4              45
1.9              40
1.5              38
1.8              36
1.6              31
2.2              31
1                30
2                26
2.5              24
2.1              23
2.4              23
2.3              23
2.7              22
3.1              20
2.6              19
3.2              18
2.8              18
3                18
2.9              17
3.3              17
3.8              17
3.5              12
4.1              12
3.4              11
4                11
4.2              11
5.2              10
3.6              10
3.7              10
6                 9
4.8               8
4.5               8
5.1               7
6.7               7
4.4               7
4.9               7
7.7               6
5.6               6
3.9 

In [12]:
# I create a detection flag for Phase I based on the raw result text.
# Values reported with "<" mean the result was below the laboratory reporting limit.
# I do not know the exact concentration for those rows. They could be zero,
# or they could be present at a very low level below the reporting limit.
# Because this project is based on any PFAS detection, I treat these rows as
# unknown (U), not as confirmed non-detects.
# Numeric values without "<" are treated as detected (Y).

phase1_long["pfas_detected"] = np.select(
    [
        phase1_long["raw_result_text"].isna(),
        phase1_long["raw_result_text"].str.startswith("<", na=False),
        phase1_long["raw_result_text"].str.match(r"^\d+(\.\d+)?$", na=False)
    ],
    [
        pd.NA,
        "U",
        "Y"
    ],
    default=pd.NA
)

In [13]:
# Phase I does not include a separate detection limit column.
# To keep the Waterkeeper data consistent with the main dataset,
# I create the column but leave it empty rather than making assumptions
# about the laboratory reporting limit.

phase1_long["DetectionQuantitationLimitMeasure/MeasureValue"] = np.nan

In [14]:
phase1_long.shape

(12312, 13)

In [15]:
print(phase1_long.columns.tolist())

['Sample ID', 'Sampling Date', 'Waterkeeper Group', 'State', 'Waterbody', 'Upstream or Downstream', 'Latitude', 'Longitude', 'CharacteristicName', 'raw_result', 'raw_result_text', 'pfas_detected', 'DetectionQuantitationLimitMeasure/MeasureValue']


In [16]:
# I standardize Waterkeeper Phase I to match data_surface.

phase1_long["MonitoringLocationIdentifier"] = phase1_long["Sample ID"]
phase1_long["ActivityStartDate"] = phase1_long["Sampling Date"]
phase1_long["OrganizationFormalName"] = phase1_long["Waterkeeper Group"]
phase1_long["state"] = phase1_long["State"]
phase1_long["MonitoringLocationName"] = phase1_long["Waterbody"]
phase1_long["ActivityLocation/LatitudeMeasure"] = phase1_long["Latitude"]
phase1_long["ActivityLocation/LongitudeMeasure"] = phase1_long["Longitude"]
phase1_long["CharacteristicName"] = phase1_long["CharacteristicName"]
phase1_long["ResultMeasureValue"] = phase1_long["raw_result"]
phase1_long["ResultMeasureValue_text"] = phase1_long["raw_result_text"]
phase1_long["pfas_detected"] = phase1_long["pfas_detected"]
phase1_long["DetectionQuantitationLimitMeasure/MeasureValue"] = phase1_long[
    "DetectionQuantitationLimitMeasure/MeasureValue"
]

phase1_long["source"] = "Waterkeeper Phase I"

In [17]:
print(phase1_long.columns.tolist())

['Sample ID', 'Sampling Date', 'Waterkeeper Group', 'State', 'Waterbody', 'Upstream or Downstream', 'Latitude', 'Longitude', 'CharacteristicName', 'raw_result', 'raw_result_text', 'pfas_detected', 'DetectionQuantitationLimitMeasure/MeasureValue', 'MonitoringLocationIdentifier', 'ActivityStartDate', 'OrganizationFormalName', 'state', 'MonitoringLocationName', 'ActivityLocation/LatitudeMeasure', 'ActivityLocation/LongitudeMeasure', 'ResultMeasureValue', 'ResultMeasureValue_text', 'source']


In [18]:
phase1_long = phase1_long[
    [
        "MonitoringLocationIdentifier",
        "MonitoringLocationName",
        "ActivityStartDate",
        "OrganizationFormalName",
        "state",
        "ActivityLocation/LatitudeMeasure",
        "ActivityLocation/LongitudeMeasure",
        "CharacteristicName",
        "ResultMeasureValue",
        "ResultMeasureValue_text",
        "DetectionQuantitationLimitMeasure/MeasureValue",
        "pfas_detected",
        "Upstream or Downstream",
        "source"
    ]
].copy()

In [19]:
# I save the standardized Phase I Waterkeeper data as a separate CSV.
# I am saving Phase I by itself so I can import it later and combine it
phase1_output_path = file_path + "saved files/waterkeeper_phase1_long.csv"

phase1_long.to_csv(
    phase1_output_path,
    index=False
)

print("Saved:", phase1_output_path)

Saved: G:/My Drive/Projects/Large projects/PFAS in water/saved files/waterkeeper_phase1_long.csv


In [20]:
# I checked manually and everything looks fine

### Dealing with Phase 2 data

In [21]:
phase2_results = pd.read_excel(
    phase2_results_path,
    sheet_name="T1 Cfree (Final)",
    header=None
)

display(phase2_results.head(15))
print(phase2_results.shape)

,0,1,2,3,4,5,6,7,8,9,...,207,208,209,210,211,212,213,214,215,216
0,Client ID,DTE Rouge W Down,NaN,NaN,DTE Rouge W Up,NaN,NaN,INLSantaAna081424-WDown,NaN,NaN,...,NaN,INL Santa Ana 081424-W UP,NaN,NaN,INL Santa Ana 081424-W Down,NaN,NaN,CAPSouthRiver 8-27-24,NaN,NaN
1,Analyte,Result,Qualifier,MDL,Result,Qualifier,MDL,Result,Qualifier,MDL,...,MDL,Result,Qualifier,MDL,Result,Qualifier,MDL,Result,Qualifier,MDL
2,NaN,(ng/L),NaN,(ng/L),(ng/L),NaN,(ng/L),(ng/L),NaN,(ng/L),...,(ng/L),(ng/L),NaN,(ng/L),(ng/L),NaN,(ng/L),(ng/L),NaN,(ng/L)
3,Perfluorobutanoic acid (PFBA),ND,NaN,1.1,5.5,NaN,1.0,12,NaN,1.4,...,5.0,13,NaN,1.1,11,NaN,1.0,3.9,J,1.1
4,Perfluoropentanoic acid (PFPeA),1.7,J I,0.53,4.3,NaN,0.52,24,NaN,0.69,...,2.5,12,NaN,0.54,43,NaN,0.52,ND,NaN,0.53
5,Perfluorohexanoic acid (PFHxA),9.2,NaN,0.53,6.2,NaN,0.52,20,NaN,0.69,...,2.5,11,NaN,0.54,27,NaN,0.52,2.2,I,0.53
6,Perfluoroheptanoic acid (PFHpA),1.6,J,0.53,1.6,J,0.52,3.6,NaN,0.69,...,2.5,3.5,NaN,0.54,2.7,NaN,0.52,1.5,J,0.53
7,Perfluorooctanoic acid (PFOA),44,NaN,0.53,3.8,NaN,0.52,13,NaN,0.69,...,2.5,11,NaN,0.54,10,NaN,0.52,3.3,NaN,0.53
8,Perfluorononanoic acid (PFNA),ND,NaN,0.54,0.61,J,0.52,2.1,J,0.69,...,2.5,1.0,J,0.54,0.73,J,0.52,1.0,J,0.53
9,Perfluorodecanoic acid (PFDA),5.0,NaN,0.54,ND,NaN,0.52,2.2,J,0.69,...,2.5,ND,NaN,0.54,ND,NaN,0.52,ND,NaN,0.53


(55, 217)


In [22]:
# I load the Phase II location file.
# This file has the sample IDs, Waterkeeper names, latitude and longitude.
# I need this later to create location names for the Phase II results.

phase2_locations = pd.read_excel(phase2_locations_path)

display(phase2_locations.head())
print(phase2_locations.shape)
print(phase2_locations.columns.tolist())

,Sample id,Licensed Waterkeeper Name,Sludge Site or Wastewater Treatment Facility,Upstream or Downstream,Deployment Date,Retrieval Date,Lat,Long,Alternative Sample ID
0,Cahaba Riverkeeper/Black Warrior,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN
1,CAHCahaba08-22-2024-WUp,Cahaba Riverkeeper and Black Warrior Riverkeeper,WWTF,Upstream,2024-08-22,2024-09-18,33.3693,-86.785329,NaN
2,CAHCahaba08-22-2024-WDown,Cahaba Riverkeeper and Black Warrior Riverkeeper,WWTF,Downstream,2024-08-22,2024-09-18,33.369614,-86.788120,NaN
3,CAHCaneCreek08-20-202024-LAUp,Cahaba Riverkeeper and Black Warrior Riverkeeper,Sludge Site,Upstream,2024-08-20,2024-09-17,33.812904,-86.836824,NaN
4,CAHCaneCreek08-20-2024-LADown,Cahaba Riverkeeper and Black Warrior Riverkeeper,Sludge Site,Downstream,2024-08-20,2024-09-17,33.792691,-86.845976,NaN


(100, 9)
['Sample id', 'Licensed Waterkeeper Name', 'Sludge Site or Wastewater Treatment Facility', 'Upstream or Downstream', 'Deployment Date', 'Retrieval Date', 'Lat ', 'Long', 'Alternative Sample ID']


In [23]:
# I clean the Phase II location file so the column names are easier to use later.

phase2_locations_clean = phase2_locations.copy()

phase2_locations_clean = phase2_locations_clean.rename(
    columns={
        "Sample id": "sample_id_location",
        "Lat ": "latitude",
        "Long": "longitude"
    }
)

# I drop the header/group rows that do not have real sample IDs or coordinates.
phase2_locations_clean = phase2_locations_clean.loc[
    phase2_locations_clean["sample_id_location"].notna()
].copy()

display(phase2_locations_clean.head())
print(phase2_locations_clean.shape)
print(phase2_locations_clean.columns.tolist())

,sample_id_location,Licensed Waterkeeper Name,Sludge Site or Wastewater Treatment Facility,Upstream or Downstream,Deployment Date,Retrieval Date,latitude,longitude,Alternative Sample ID
0,Cahaba Riverkeeper/Black Warrior,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN
1,CAHCahaba08-22-2024-WUp,Cahaba Riverkeeper and Black Warrior Riverkeeper,WWTF,Upstream,2024-08-22,2024-09-18,33.3693,-86.785329,NaN
2,CAHCahaba08-22-2024-WDown,Cahaba Riverkeeper and Black Warrior Riverkeeper,WWTF,Downstream,2024-08-22,2024-09-18,33.369614,-86.788120,NaN
3,CAHCaneCreek08-20-202024-LAUp,Cahaba Riverkeeper and Black Warrior Riverkeeper,Sludge Site,Upstream,2024-08-20,2024-09-17,33.812904,-86.836824,NaN
4,CAHCaneCreek08-20-2024-LADown,Cahaba Riverkeeper and Black Warrior Riverkeeper,Sludge Site,Downstream,2024-08-20,2024-09-17,33.792691,-86.845976,NaN


(100, 9)
['sample_id_location', 'Licensed Waterkeeper Name', 'Sludge Site or Wastewater Treatment Facility', 'Upstream or Downstream', 'Deployment Date', 'Retrieval Date', 'latitude', 'longitude', 'Alternative Sample ID']


In [24]:
# I look at the raw Phase II results file structure.
# This helps me see where the real headers and sample IDs are.

display(phase2_results.head(20))
print(phase2_results.shape)

,0,1,2,3,4,5,6,7,8,9,...,207,208,209,210,211,212,213,214,215,216
0,Client ID,DTE Rouge W Down,NaN,NaN,DTE Rouge W Up,NaN,NaN,INLSantaAna081424-WDown,NaN,NaN,...,NaN,INL Santa Ana 081424-W UP,NaN,NaN,INL Santa Ana 081424-W Down,NaN,NaN,CAPSouthRiver 8-27-24,NaN,NaN
1,Analyte,Result,Qualifier,MDL,Result,Qualifier,MDL,Result,Qualifier,MDL,...,MDL,Result,Qualifier,MDL,Result,Qualifier,MDL,Result,Qualifier,MDL
2,NaN,(ng/L),NaN,(ng/L),(ng/L),NaN,(ng/L),(ng/L),NaN,(ng/L),...,(ng/L),(ng/L),NaN,(ng/L),(ng/L),NaN,(ng/L),(ng/L),NaN,(ng/L)
3,Perfluorobutanoic acid (PFBA),ND,NaN,1.1,5.5,NaN,1.0,12,NaN,1.4,...,5.0,13,NaN,1.1,11,NaN,1.0,3.9,J,1.1
4,Perfluoropentanoic acid (PFPeA),1.7,J I,0.53,4.3,NaN,0.52,24,NaN,0.69,...,2.5,12,NaN,0.54,43,NaN,0.52,ND,NaN,0.53
5,Perfluorohexanoic acid (PFHxA),9.2,NaN,0.53,6.2,NaN,0.52,20,NaN,0.69,...,2.5,11,NaN,0.54,27,NaN,0.52,2.2,I,0.53
6,Perfluoroheptanoic acid (PFHpA),1.6,J,0.53,1.6,J,0.52,3.6,NaN,0.69,...,2.5,3.5,NaN,0.54,2.7,NaN,0.52,1.5,J,0.53
7,Perfluorooctanoic acid (PFOA),44,NaN,0.53,3.8,NaN,0.52,13,NaN,0.69,...,2.5,11,NaN,0.54,10,NaN,0.52,3.3,NaN,0.53
8,Perfluorononanoic acid (PFNA),ND,NaN,0.54,0.61,J,0.52,2.1,J,0.69,...,2.5,1.0,J,0.54,0.73,J,0.52,1.0,J,0.53
9,Perfluorodecanoic acid (PFDA),5.0,NaN,0.54,ND,NaN,0.52,2.2,J,0.69,...,2.5,ND,NaN,0.54,ND,NaN,0.52,ND,NaN,0.53


(55, 217)


In [25]:
# I clean the raw Phase II results file into a long format.
# The raw file has sample IDs across the columns, with Result, Qualifier and MDL repeated for each sample.

phase2_raw = phase2_results.copy()

# Row 0 has the sample IDs.
sample_id_row = phase2_raw.iloc[0]

# Row 1 has Result / Qualifier / MDL.
measure_type_row = phase2_raw.iloc[1]

# The real PFAS analyte rows start at row 3.
phase2_data_rows = phase2_raw.iloc[3:].copy()

# The first column is the PFAS chemical name.
phase2_data_rows = phase2_data_rows.rename(columns={0: "CharacteristicName"})

long_blocks = []

# Starting from column 1, every sample uses 3 columns:
# Result, Qualifier, MDL.
for col in range(1, phase2_raw.shape[1], 3):
    
    sample_id = sample_id_row[col]
    
    # I skip empty sample IDs.
    if pd.isna(sample_id):
        continue
    
    temp = phase2_data_rows[
        ["CharacteristicName", col, col + 1, col + 2]
    ].copy()
    
    temp.columns = [
        "CharacteristicName",
        "raw_result",
        "MeasureQualifierCode",
        "MDL"
    ]
    
    temp["sample_id"] = sample_id
    
    long_blocks.append(temp)

phase2_long = pd.concat(long_blocks, ignore_index=True)

display(phase2_long.head())
print(phase2_long.shape)
print(phase2_long.columns.tolist())

,CharacteristicName,raw_result,MeasureQualifierCode,MDL,sample_id
0,Perfluorobutanoic acid (PFBA),ND,NaN,1.1,DTE Rouge W Down
1,Perfluoropentanoic acid (PFPeA),1.7,J I,0.53,DTE Rouge W Down
2,Perfluorohexanoic acid (PFHxA),9.2,NaN,0.53,DTE Rouge W Down
3,Perfluoroheptanoic acid (PFHpA),1.6,J,0.53,DTE Rouge W Down
4,Perfluorooctanoic acid (PFOA),44,NaN,0.53,DTE Rouge W Down


(3744, 5)
['CharacteristicName', 'raw_result', 'MeasureQualifierCode', 'MDL', 'sample_id']


In [26]:
# I compare the sample IDs from Phase II results with the sample IDs from Phase II locations.

phase2_results_ids = (
    phase2_long["sample_id"]
    .dropna()
    .drop_duplicates()
    .sort_values()
)

phase2_location_ids = (
    phase2_locations_clean["sample_id_location"]
    .dropna()
    .drop_duplicates()
    .sort_values()
)

print("Unique Phase II result sample IDs:", phase2_results_ids.nunique())
print("Unique Phase II location sample IDs:", phase2_location_ids.nunique())

display(pd.DataFrame({
    "phase2_result_sample_ids": phase2_results_ids.reset_index(drop=True),
    "phase2_location_sample_ids": phase2_location_ids.reset_index(drop=True)
}))

Unique Phase II result sample IDs: 72
Unique Phase II location sample IDs: 99


,phase2_result_sample_ids,phase2_location_sample_ids
0,BAYHuntingBayou[08/21/2024]-Wdown,\nLOSLARiver080824-WDown
1,BAYHuntingBayou[08/21/2024]-Wup,\nLOSLARiver080824-WUp
2,CAHCahaba 08-22-2024-WDown,BAYHuntingBayou[08/21/2024]-WDown
3,CAHCahaba 08-22-2024-WUp,BAYHuntingBayou[08/21/2024]-WUp
4,CAHCaneCreek 08-20-2024-LADown,BLAPocotaligo[8/23/2024]-WDown
...,...,...
94,NaN,UPROpequon-10/11//2024WUp
95,NaN,Upper Potomac Riverkeeper
96,NaN,WESOhioRiver[8/12/2024]-WUp
97,NaN,WESOhioRiver[9/11/2024]-WDown


In [27]:
# I remove rows from the Phase II location file that are group/header rows, not real sample locations.
# Real sample rows should have latitude and longitude.

phase2_locations_clean = phase2_locations_clean.loc[
    phase2_locations_clean["latitude"].notna()
    & phase2_locations_clean["longitude"].notna()
].copy()

print("Cleaned Phase II location rows:", phase2_locations_clean.shape)

display(
    phase2_locations_clean[
        [
            "sample_id_location",
            "Licensed Waterkeeper Name",
            "latitude",
            "longitude",
            "Alternative Sample ID"
        ]
    ].head(20)
)

Cleaned Phase II location rows: (73, 9)


,sample_id_location,Licensed Waterkeeper Name,latitude,longitude,Alternative Sample ID
1,CAHCahaba08-22-2024-WUp,Cahaba Riverkeeper and Black Warrior Riverkeeper,33.3693,-86.785329,NaN
2,CAHCahaba08-22-2024-WDown,Cahaba Riverkeeper and Black Warrior Riverkeeper,33.369614,-86.788120,NaN
3,CAHCaneCreek08-20-202024-LAUp,Cahaba Riverkeeper and Black Warrior Riverkeeper,33.812904,-86.836824,NaN
4,CAHCaneCreek08-20-2024-LADown,Cahaba Riverkeeper and Black Warrior Riverkeeper,33.792691,-86.845976,NaN
5,CAHCaneCreek08-20-202024-LAUp-DUP,Cahaba Riverkeeper and Black Warrior Riverkeeper,33.812904,-86.836824,NaN
6,CAHCaneCreek08-20-2024-LADown-DUP,Cahaba Riverkeeper and Black Warrior Riverkeeper,33.792691,-86.845976,NaN
8,COOBigWills[08/08/02024]-WUp,Coosa Riverkeeper,33.988814,-86.037728,NaN
9,COOBigWills[08/08/02024]-WDown,Coosa Riverkeeper,33.981597,-86.030581,NaN
10,COOWhippoorwill[08/14/2024]-LAUp,Coosa Riverkeeper,34.043134,-86.338660,NaN
11,COOWhippoorwill[08/14/2024]-LADown,Coosa Riverkeeper,34.064041,-86.357335,NaN


In [28]:
# I compare real Phase II result sample IDs with real Phase II location sample IDs.

phase2_results_ids = (
    phase2_long["sample_id"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

phase2_location_ids = (
    phase2_locations_clean["sample_id_location"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

print("Unique Phase II result sample IDs:", phase2_results_ids.nunique())
print("Unique real Phase II location sample IDs:", phase2_location_ids.nunique())

comparison_length = max(len(phase2_results_ids), len(phase2_location_ids))

comparison = pd.DataFrame({
    "phase2_result_sample_ids": phase2_results_ids.reindex(range(comparison_length)),
    "phase2_location_sample_ids": phase2_location_ids.reindex(range(comparison_length))
})

display(comparison)

Unique Phase II result sample IDs: 72
Unique real Phase II location sample IDs: 72


,phase2_result_sample_ids,phase2_location_sample_ids
0,BAYHuntingBayou[08/21/2024]-Wdown,\nLOSLARiver080824-WDown
1,BAYHuntingBayou[08/21/2024]-Wup,\nLOSLARiver080824-WUp
2,CAHCahaba 08-22-2024-WDown,BAYHuntingBayou[08/21/2024]-WDown
3,CAHCahaba 08-22-2024-WUp,BAYHuntingBayou[08/21/2024]-WUp
4,CAHCaneCreek 08-20-2024-LADown,BLAPocotaligo[8/23/2024]-WDown
...,...,...
67,UPR BackCreek 8/6/24 LA Up (DUP),UPRBackCreek8/6/24-LAUp-DUP
68,UPRPFAS 1,UPROpequon- 10/11/2024WDown
69,UPRPFAS 2,UPROpequon-10/11//2024WUp
70,WESOhioRiver 8-12-24-WDown,WESOhioRiver[8/12/2024]-WUp


In [29]:
# I export all unique Phase II result sample IDs for manual matching.
# Tony this is new and I changed the manual file
result_sample_ids = (
    phase2_long["sample_id"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

manual_matching = pd.DataFrame({
    "sample_id": result_sample_ids,
    "MonitoringLocationName": ""
})

manual_matching.to_csv(
    file_path + "saved files/phase2_manual_matching.csv",
    index=False
)

print("Saved:")
print(file_path + "saved files/phase2_manual_matching.csv")

Saved:
G:/My Drive/Projects/Large projects/PFAS in water/saved files/phase2_manual_matching.csv


In [30]:
#NEW
# I manually added name based on google map where you can see the links in the file and manually added lat and long based on WA location
# I load the manually fixed Phase II location names.

fixed_names = pd.read_excel(
    file_path + "saved files/phase2_locations_for_manual_review_manual_edit TD comments (1).xlsx"
)

fixed_names = fixed_names.drop_duplicates().copy()

display(fixed_names.head())
print(fixed_names.shape)
print(fixed_names.columns.tolist())

,Sample ID,Latitude,Longitude,Upstream or Downstream,waterway,map link,Unnamed: 6
0,DTE Rouge W Down,42.281019,-83.128186,Downstream,Rouge River Downstream,https://www.google.com/maps/place/42%C2%B016'5...,NaN
1,DTE Rouge W Up,42.286553,-83.139933,Upstream,Rouge River Upstream,NaN,NaN
2,INLSantaAna081424-WDown,33.962767,-117.472040,Downstream,Santa Ana River Downstream,https://www.google.com/maps/place/33%C2%B057'4...,NaN
3,NARP Pawtuxet - 8/16/24 - WUp,41.750934,-71.445173,Upstream,Pawtuxet River Upstream,https://www.google.com/maps/place/41%C2%B045'0...,NaN
4,NARP Pawtuxet - 8/16/24 - WUp DUP,41.750934,-71.445173,Upstream,Pawtuxet River Upstream,NaN,NaN


(72, 7)
['Sample ID', 'Latitude', 'Longitude', 'Upstream or Downstream', 'waterway ', 'map link', 'Unnamed: 6']


In [31]:
print(phase2_long.columns.tolist())

['CharacteristicName', 'raw_result', 'MeasureQualifierCode', 'MDL', 'sample_id']


In [32]:
# ok now merging them
# I clean the fixed Phase II location file before merging.

fixed_names.columns = fixed_names.columns.str.strip()

fixed_names = fixed_names.drop(
    columns=["Unnamed: 6"],
    errors="ignore"
)

fixed_names = fixed_names.drop_duplicates(
    subset=["Sample ID"]
).copy()

display(fixed_names.head())
print(fixed_names.shape)
print(fixed_names.columns.tolist())

,Sample ID,Latitude,Longitude,Upstream or Downstream,waterway,map link
0,DTE Rouge W Down,42.281019,-83.128186,Downstream,Rouge River Downstream,https://www.google.com/maps/place/42%C2%B016'5...
1,DTE Rouge W Up,42.286553,-83.139933,Upstream,Rouge River Upstream,NaN
2,INLSantaAna081424-WDown,33.962767,-117.472040,Downstream,Santa Ana River Downstream,https://www.google.com/maps/place/33%C2%B057'4...
3,NARP Pawtuxet - 8/16/24 - WUp,41.750934,-71.445173,Upstream,Pawtuxet River Upstream,https://www.google.com/maps/place/41%C2%B045'0...
4,NARP Pawtuxet - 8/16/24 - WUp DUP,41.750934,-71.445173,Upstream,Pawtuxet River Upstream,NaN


(72, 6)
['Sample ID', 'Latitude', 'Longitude', 'Upstream or Downstream', 'waterway', 'map link']


In [33]:
# I merge the manually fixed Phase II location names and coordinates
# onto the Phase II long data.

phase2_long = phase2_long.merge(
    fixed_names[
        [
            "Sample ID",
            "Latitude",
            "Longitude",
            "Upstream or Downstream",
            "waterway",
            "map link"
        ]
    ],
    left_on="sample_id",
    right_on="Sample ID",
    how="left",
    validate="many_to_one"
)

display(phase2_long.head())
print(phase2_long.shape)
print(phase2_long.columns.tolist())

,CharacteristicName,raw_result,MeasureQualifierCode,MDL,sample_id,Sample ID,Latitude,Longitude,Upstream or Downstream,waterway,map link
0,Perfluorobutanoic acid (PFBA),ND,NaN,1.1,DTE Rouge W Down,DTE Rouge W Down,42.281019,-83.128186,Downstream,Rouge River Downstream,https://www.google.com/maps/place/42%C2%B016'5...
1,Perfluoropentanoic acid (PFPeA),1.7,J I,0.53,DTE Rouge W Down,DTE Rouge W Down,42.281019,-83.128186,Downstream,Rouge River Downstream,https://www.google.com/maps/place/42%C2%B016'5...
2,Perfluorohexanoic acid (PFHxA),9.2,NaN,0.53,DTE Rouge W Down,DTE Rouge W Down,42.281019,-83.128186,Downstream,Rouge River Downstream,https://www.google.com/maps/place/42%C2%B016'5...
3,Perfluoroheptanoic acid (PFHpA),1.6,J,0.53,DTE Rouge W Down,DTE Rouge W Down,42.281019,-83.128186,Downstream,Rouge River Downstream,https://www.google.com/maps/place/42%C2%B016'5...
4,Perfluorooctanoic acid (PFOA),44,NaN,0.53,DTE Rouge W Down,DTE Rouge W Down,42.281019,-83.128186,Downstream,Rouge River Downstream,https://www.google.com/maps/place/42%C2%B016'5...


(3744, 11)
['CharacteristicName', 'raw_result', 'MeasureQualifierCode', 'MDL', 'sample_id', 'Sample ID', 'Latitude', 'Longitude', 'Upstream or Downstream', 'waterway', 'map link']


In [34]:
# I check Phase II samples that did not match the manual location file.

missing_phase2_locations = phase2_long[
    phase2_long["Latitude"].isna()
    | phase2_long["Longitude"].isna()
]

display(
    missing_phase2_locations[
        ["sample_id", "CharacteristicName", "raw_result"]
    ].drop_duplicates()
)

print(
    "Unique sample IDs without location match:",
    missing_phase2_locations["sample_id"].nunique()
)

,sample_id,CharacteristicName,raw_result


Unique sample IDs without location match: 0


In [35]:
# I clean the Phase II result text.
# Phase II already has one row per sample and PFAS chemical after reshaping.

phase2_long["raw_result_text"] = (
    phase2_long["raw_result"]
    .astype("string")
    .str.strip()
)

display(
    phase2_long["raw_result_text"]
    .value_counts(dropna=False)
    .head(50)
)

raw_result_text
ND       2264
<NA>      844
1.5        20
1.3        16
1.7        16
3.1        15
Notes      14
1.1        14
2.2        14
2.7        13
11         12
2.1        12
2.5        11
3.5        11
1.6        10
2.6        10
2.0         9
1.9         9
1.2         9
4.2         8
3.9         8
2.3         8
3.7         8
1.4         8
1.0         8
3.3         8
5.5         8
12          8
1.8         7
3.2         7
6.7         6
2.8         6
4.4         6
5.2         6
4.7         6
4.9         6
7.1         6
13          6
4.5         6
3.4         5
5.7         5
3.6         5
4.3         5
6.1         5
6.2         5
4.1         5
4.8         5
5.8         5
10          4
0.54        4
Name: count, dtype: Int64

In [36]:
# I review rows where the result is missing or contains Notes.

phase2_long.loc[
    phase2_long["raw_result_text"].isin(["Notes"]) |
    phase2_long["raw_result_text"].isna(),
    [
        "CharacteristicName",
        "raw_result",
        "MeasureQualifierCode",
        "MDL",
        "sample_id"
    ]
]

,CharacteristicName,raw_result,MeasureQualifierCode,MDL,sample_id
40,NaN,Notes,MDL: Method detection limit,NaN,DTE Rouge W Down
41,NaN,NaN,ng/L: Nanograms per liter,NaN,DTE Rouge W Down
42,NaN,NaN,ND: Not detected at the MDL shown in the third...,NaN,DTE Rouge W Down
43,NaN,NaN,B: Compounds was found in the blank and sample.,NaN,DTE Rouge W Down
45,NaN,NaN,*-: LCS and/or LCSD is outside acceptance limi...,NaN,DTE Rouge W Down
...,...,...,...,...,...
3739,NaN,NaN,NaN,NaN,CAPSouthRiver 8-27-24
3740,NaN,NaN,NaN,NaN,CAPSouthRiver 8-27-24
3741,NaN,NaN,NaN,NaN,CAPSouthRiver 8-27-24
3742,NaN,NaN,NaN,NaN,CAPSouthRiver 8-27-24


In [37]:
# I keep only the real PFAS analyte rows.
# The rows after the analyte list include notes or blank rows, so I remove them here.

phase2_long = phase2_long.loc[
    phase2_long["CharacteristicName"].notna()
].copy()

phase2_long = phase2_long.loc[
    ~phase2_long["CharacteristicName"].astype(str).str.contains(
        "note|notes|qualifier|client|analyte",
        case=False,
        na=False
    )
].copy()

display(phase2_long["CharacteristicName"].value_counts(dropna=False))
print(phase2_long.shape)

CharacteristicName
Perfluorobutanoic acid (PFBA)                  72
Perfluoropentanoic acid (PFPeA)                72
Perfluorohexanoic acid (PFHxA)                 72
Perfluoroheptanoic acid (PFHpA)                72
Perfluorooctanoic acid (PFOA)                  72
Perfluorononanoic acid (PFNA)                  72
Perfluorodecanoic acid (PFDA)                  72
Perfluoroundecanoic acid (PFUnA)               72
Perfluorododecanoic acid (PFDoA)               72
Perfluorotridecanoic acid (PFTrDA)             72
Perfluorotetradecanoic acid (PFTeA)            72
Perfluorobutanesulfonic acid (PFBS)            72
Perfluoropentanesulfonic acid (PFPeS)          72
Perfluorohexanesulfonic acid (PFHxS)           72
Perfluoroheptanesulfonic acid (PFHpS)          72
Perfluorooctanesulfonic acid (PFOS)            72
Perfluorononanesulfonic acid (PFNS)            72
Perfluorodecanesulfonic acid (PFDS)            72
Perfluorododecanesulfonic acid (PFDoS)         72
4:2 FTS                        

(2880, 12)


In [38]:
# I check the result values again after removing notes/non-analyte rows.

phase2_long["raw_result_text"] = (
    phase2_long["raw_result"]
    .astype("string")
    .str.strip()
)

display(
    phase2_long["raw_result_text"]
    .value_counts(dropna=False)
    .head(50)
)

raw_result_text
ND      2264
1.5       20
1.3       16
1.7       16
3.1       15
2.2       14
1.1       14
2.7       13
11        12
2.1       12
3.5       11
2.5       11
2.6       10
1.6       10
1.9        9
2.0        9
1.2        9
12         8
1.4        8
5.5        8
3.3        8
3.9        8
4.2        8
1.0        8
3.7        8
2.3        8
3.2        7
1.8        7
5.2        6
6.7        6
2.8        6
4.4        6
4.5        6
4.7        6
4.9        6
13         6
7.1        6
6.2        5
3.6        5
4.3        5
3.4        5
5.7        5
6.1        5
5.8        5
4.1        5
4.8        5
5.0        4
3.8        4
0.54       4
0.62       4
Name: count, dtype: Int64

In [39]:
phase2_long.loc[
    phase2_long["sample_id"] == "DTE Rouge W Down"
].head(20)

,CharacteristicName,raw_result,MeasureQualifierCode,MDL,sample_id,Sample ID,Latitude,Longitude,Upstream or Downstream,waterway,map link,raw_result_text
0,Perfluorobutanoic acid (PFBA),ND,NaN,1.1,DTE Rouge W Down,DTE Rouge W Down,42.281019,-83.128186,Downstream,Rouge River Downstream,https://www.google.com/maps/place/42%C2%B016'5...,ND
1,Perfluoropentanoic acid (PFPeA),1.7,J I,0.53,DTE Rouge W Down,DTE Rouge W Down,42.281019,-83.128186,Downstream,Rouge River Downstream,https://www.google.com/maps/place/42%C2%B016'5...,1.7
2,Perfluorohexanoic acid (PFHxA),9.2,NaN,0.53,DTE Rouge W Down,DTE Rouge W Down,42.281019,-83.128186,Downstream,Rouge River Downstream,https://www.google.com/maps/place/42%C2%B016'5...,9.2
3,Perfluoroheptanoic acid (PFHpA),1.6,J,0.53,DTE Rouge W Down,DTE Rouge W Down,42.281019,-83.128186,Downstream,Rouge River Downstream,https://www.google.com/maps/place/42%C2%B016'5...,1.6
4,Perfluorooctanoic acid (PFOA),44,NaN,0.53,DTE Rouge W Down,DTE Rouge W Down,42.281019,-83.128186,Downstream,Rouge River Downstream,https://www.google.com/maps/place/42%C2%B016'5...,44
5,Perfluorononanoic acid (PFNA),ND,NaN,0.54,DTE Rouge W Down,DTE Rouge W Down,42.281019,-83.128186,Downstream,Rouge River Downstream,https://www.google.com/maps/place/42%C2%B016'5...,ND
6,Perfluorodecanoic acid (PFDA),5.0,NaN,0.54,DTE Rouge W Down,DTE Rouge W Down,42.281019,-83.128186,Downstream,Rouge River Downstream,https://www.google.com/maps/place/42%C2%B016'5...,5.0
7,Perfluoroundecanoic acid (PFUnA),ND,NaN,0.54,DTE Rouge W Down,DTE Rouge W Down,42.281019,-83.128186,Downstream,Rouge River Downstream,https://www.google.com/maps/place/42%C2%B016'5...,ND
8,Perfluorododecanoic acid (PFDoA),ND,NaN,0.59,DTE Rouge W Down,DTE Rouge W Down,42.281019,-83.128186,Downstream,Rouge River Downstream,https://www.google.com/maps/place/42%C2%B016'5...,ND
9,Perfluorotridecanoic acid (PFTrDA),ND,NaN,0.63,DTE Rouge W Down,DTE Rouge W Down,42.281019,-83.128186,Downstream,Rouge River Downstream,https://www.google.com/maps/place/42%C2%B016'5...,ND


In [40]:
# I clean the Phase II result values and create numeric result values.

phase2_long["raw_result_text"] = (
    phase2_long["raw_result"]
    .astype("string")
    .str.strip()
)

phase2_long["ResultMeasureValue_num"] = pd.to_numeric(
    phase2_long["raw_result_text"],
    errors="coerce"
)

display(phase2_long[["raw_result_text", "ResultMeasureValue_num"]].head(20))

,raw_result_text,ResultMeasureValue_num
0,ND,<NA>
1,1.7,1.7
2,9.2,9.2
3,1.6,1.6
4,44,44.0
5,ND,<NA>
6,5.0,5.0
7,ND,<NA>
8,ND,<NA>
9,ND,<NA>


In [41]:
# these files are clear if ND it means no PFAS
# I create the PFAS detection flag for Phase II.
# ND means not detected. Numeric values mean detected.

phase2_long["pfas_detected"] = np.where(
    phase2_long["raw_result_text"].str.upper().eq("ND"),
    "N",
    np.where(
        phase2_long["ResultMeasureValue_num"].notna(),
        "Y",
        pd.NA
    )
)

display(phase2_long["pfas_detected"].value_counts(dropna=False))

pfas_detected
N    2264
Y     616
Name: count, dtype: int64

In [42]:
# I review rows where PFAS detection is still unclear.

phase2_unclear_results = phase2_long.loc[
    phase2_long["pfas_detected"].isna(),
    [
        "sample_id",
        "CharacteristicName",
        "raw_result",
        "raw_result_text",
        "MeasureQualifierCode",
        "MDL"
    ]
]

display(phase2_unclear_results)
print("Unclear rows:", len(phase2_unclear_results))

,sample_id,CharacteristicName,raw_result,raw_result_text,MeasureQualifierCode,MDL


Unclear rows: 0


In [43]:
print(phase2_long.shape)
print(phase2_long.columns.tolist())

(2880, 14)
['CharacteristicName', 'raw_result', 'MeasureQualifierCode', 'MDL', 'sample_id', 'Sample ID', 'Latitude', 'Longitude', 'Upstream or Downstream', 'waterway', 'map link', 'raw_result_text', 'ResultMeasureValue_num', 'pfas_detected']


In [44]:
# I check the Phase I columns.

print(phase1_long.shape)
print(phase1_long.columns.tolist())

(12312, 14)
['MonitoringLocationIdentifier', 'MonitoringLocationName', 'ActivityStartDate', 'OrganizationFormalName', 'state', 'ActivityLocation/LatitudeMeasure', 'ActivityLocation/LongitudeMeasure', 'CharacteristicName', 'ResultMeasureValue', 'ResultMeasureValue_text', 'DetectionQuantitationLimitMeasure/MeasureValue', 'pfas_detected', 'Upstream or Downstream', 'source']


In [45]:
# I rename the Phase II columns so they match the Phase I structure.
# I standardize Waterkeeper Phase II to match Phase I.

phase2_long["MonitoringLocationIdentifier"] = phase2_long["sample_id"]
phase2_long["MonitoringLocationName"] = phase2_long["waterway"]
phase2_long["ActivityStartDate"] = pd.NA
phase2_long["OrganizationFormalName"] = "Waterkeeper Alliance"
phase2_long["state"] = pd.NA

phase2_long["ActivityLocation/LatitudeMeasure"] = phase2_long["Latitude"]
phase2_long["ActivityLocation/LongitudeMeasure"] = phase2_long["Longitude"]

phase2_long["ResultMeasureValue"] = phase2_long["ResultMeasureValue_num"]
phase2_long["ResultMeasureValue_text"] = phase2_long["raw_result_text"]

phase2_long["DetectionQuantitationLimitMeasure/MeasureValue"] = phase2_long["MDL"]

phase2_long["source"] = "Waterkeeper Phase II"

In [46]:
# I keep only the standardized columns so Phase II matches Phase I.

phase2_long = phase2_long[
    [
        "MonitoringLocationIdentifier",
        "MonitoringLocationName",
        "ActivityStartDate",
        "OrganizationFormalName",
        "state",
        "ActivityLocation/LatitudeMeasure",
        "ActivityLocation/LongitudeMeasure",
        "CharacteristicName",
        "ResultMeasureValue",
        "ResultMeasureValue_text",
        "DetectionQuantitationLimitMeasure/MeasureValue",
        "pfas_detected",
        "Upstream or Downstream",
        "source"
    ]
].copy()

print(phase2_long.shape)
print(phase2_long.columns.tolist())

(2880, 14)
['MonitoringLocationIdentifier', 'MonitoringLocationName', 'ActivityStartDate', 'OrganizationFormalName', 'state', 'ActivityLocation/LatitudeMeasure', 'ActivityLocation/LongitudeMeasure', 'CharacteristicName', 'ResultMeasureValue', 'ResultMeasureValue_text', 'DetectionQuantitationLimitMeasure/MeasureValue', 'pfas_detected', 'Upstream or Downstream', 'source']


In [47]:
# I check the states in the Phase I file.

display(
    phase2_long["state"]
    .value_counts(dropna=False)
    .sort_index()
)

state
<NA>    2880
Name: count, dtype: int64

In [48]:
# I check the states in the Phase I file.

display(
    phase1_long["state"]
    .value_counts(dropna=False)
    .sort_index()
)

state
AK       108
AL       864
AR       108
CA      1188
CO       216
CT       108
D.C.     108
FL       810
GA       972
ID       216
KS       108
KY       108
MA       108
MD      1080
ME       108
MI       324
MO       108
MS       108
MT       216
NC      1080
NJ       108
NY       540
OH       162
OK       324
OR       216
PA       432
RI       324
SC       594
TN       108
TX       216
VA       324
VT       108
WA       540
WI       108
WV       162
Name: count, dtype: int64

In [49]:
print(phase1_long.columns.tolist())

['MonitoringLocationIdentifier', 'MonitoringLocationName', 'ActivityStartDate', 'OrganizationFormalName', 'state', 'ActivityLocation/LatitudeMeasure', 'ActivityLocation/LongitudeMeasure', 'CharacteristicName', 'ResultMeasureValue', 'ResultMeasureValue_text', 'DetectionQuantitationLimitMeasure/MeasureValue', 'pfas_detected', 'Upstream or Downstream', 'source']


In [50]:
print(phase2_long.columns.tolist())

['MonitoringLocationIdentifier', 'MonitoringLocationName', 'ActivityStartDate', 'OrganizationFormalName', 'state', 'ActivityLocation/LatitudeMeasure', 'ActivityLocation/LongitudeMeasure', 'CharacteristicName', 'ResultMeasureValue', 'ResultMeasureValue_text', 'DetectionQuantitationLimitMeasure/MeasureValue', 'pfas_detected', 'Upstream or Downstream', 'source']


In [51]:
print("Phase I unique states:")
print(phase1_long["state"].nunique(dropna=True))

display(
    phase1_long["state"]
    .value_counts(dropna=False)
    .sort_index()
)

Phase I unique states:
35


state
AK       108
AL       864
AR       108
CA      1188
CO       216
CT       108
D.C.     108
FL       810
GA       972
ID       216
KS       108
KY       108
MA       108
MD      1080
ME       108
MI       324
MO       108
MS       108
MT       216
NC      1080
NJ       108
NY       540
OH       162
OK       324
OR       216
PA       432
RI       324
SC       594
TN       108
TX       216
VA       324
VT       108
WA       540
WI       108
WV       162
Name: count, dtype: int64

In [52]:
print("Phase II unique states:")
print(phase2_long["state"].nunique(dropna=True))

display(
    phase2_long["state"]
    .value_counts(dropna=False)
    .sort_index()
)

Phase II unique states:
0


state
<NA>    2880
Name: count, dtype: int64

In [53]:
# I clean latitude and longitude before creating point geometry.
# Some values have extra commas or spaces, so I remove them first.

phase2_long["ActivityLocation/LatitudeMeasure"] = (
    phase2_long["ActivityLocation/LatitudeMeasure"]
    .astype("string")
    .str.replace(",", "", regex=False)
    .str.strip()
)

phase2_long["ActivityLocation/LongitudeMeasure"] = (
    phase2_long["ActivityLocation/LongitudeMeasure"]
    .astype("string")
    .str.replace(",", "", regex=False)
    .str.strip()
)

phase2_long["ActivityLocation/LatitudeMeasure"] = pd.to_numeric(
    phase2_long["ActivityLocation/LatitudeMeasure"],
    errors="coerce"
)

phase2_long["ActivityLocation/LongitudeMeasure"] = pd.to_numeric(
    phase2_long["ActivityLocation/LongitudeMeasure"],
    errors="coerce"
)

print(
    phase2_long[
        [
            "ActivityLocation/LatitudeMeasure",
            "ActivityLocation/LongitudeMeasure"
        ]
    ]
    .dtypes
)

ActivityLocation/LatitudeMeasure     Float64
ActivityLocation/LongitudeMeasure    Float64
dtype: object


In [54]:
print(
    "Missing latitude:",
    phase2_long["ActivityLocation/LatitudeMeasure"].isna().sum()
)

print(
    "Missing longitude:",
    phase2_long["ActivityLocation/LongitudeMeasure"].isna().sum()
)

display(
    phase2_long.loc[
        phase2_long["ActivityLocation/LatitudeMeasure"].isna()
        | phase2_long["ActivityLocation/LongitudeMeasure"].isna(),
        [
            "MonitoringLocationIdentifier",
            "MonitoringLocationName",
            "ActivityLocation/LatitudeMeasure",
            "ActivityLocation/LongitudeMeasure"
        ]
    ].drop_duplicates()
)

Missing latitude: 0
Missing longitude: 0


,MonitoringLocationIdentifier,MonitoringLocationName,ActivityLocation/LatitudeMeasure,ActivityLocation/LongitudeMeasure


In [55]:
import geopandas as gpd

# I load U.S. state boundaries.

states_gdf = gpd.read_file(
    "https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_state_500k.zip"
)[["STUSPS", "geometry"]]

# I make sure the state boundary CRS matches the Phase II point CRS.

states_gdf = states_gdf.to_crs("EPSG:4326")

# I create point geometries from the Phase II coordinates.

phase2_gdf = gpd.GeoDataFrame(
    phase2_long.copy(),
    geometry=gpd.points_from_xy(
        phase2_long["ActivityLocation/LongitudeMeasure"],
        phase2_long["ActivityLocation/LatitudeMeasure"]
    ),
    crs="EPSG:4326"
)

# I assign states using a spatial join.

phase2_gdf = gpd.sjoin(
    phase2_gdf,
    states_gdf,
    how="left",
    predicate="within"
)

# I store the state postal code.

phase2_gdf["state"] = phase2_gdf["STUSPS"]

phase2_long = pd.DataFrame(
    phase2_gdf.drop(
        columns=[
            "geometry",
            "index_right",
            "STUSPS"
        ],
        errors="ignore"
    )
)

print("Unique states:", phase2_long["state"].nunique())
display(phase2_long["state"].value_counts(dropna=False))

Unique states: 20


state
AL    400
NC    360
WV    280
WI    240
CA    200
RI    160
WA    160
MD    160
VA    120
MS    120
MI     80
GA     80
CT     80
OR     80
TX     80
NY     80
SC     80
OH     40
FL     40
NJ     40
Name: count, dtype: int64

In [56]:
display(
    phase2_long["state"]
    .value_counts(dropna=False)
    .sort_index()
)

print("Rows still missing state:", phase2_long["state"].isna().sum())

state
AL    400
CA    200
CT     80
FL     40
GA     80
MD    160
MI     80
MS    120
NC    360
NJ     40
NY     80
OH     40
OR     80
RI    160
SC     80
TX     80
VA    120
WA    160
WI    240
WV    280
Name: count, dtype: int64

Rows still missing state: 0


In [57]:
# I save the cleaned Phase II long file.

output_path = file_path + "saved files/phase2_long.csv"

phase2_long.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)

Saved: G:/My Drive/Projects/Large projects/PFAS in water/saved files/phase2_long.csv


### MA data


Downloaded from
https://www.mass.gov/info-details/pfas-in-surface-water-and-biota#pfas-in-surface-water-freshwater-fish-and-invertebrates-and-coastal-fish-and-shellfish-2024

In [58]:
# I load the Massachusetts PFAS metadata file.

ma_metadata_path = (
    file_path
    + "raw_data/MA data/PFAS 2024 Results_04-10-2026-a11y.xlsx"
)

ma_metadata_2024 = pd.read_excel(
    ma_metadata_path,
    sheet_name="Water_RegN"
)

display(ma_metadata_2024.head())
print(ma_metadata_2024.shape)
print(ma_metadata_2024.columns.tolist())

,Project sampleID,Site Name,SiteID,Phase,Freshwater or Coastal,Matrix,"Sample Type (G,C,I)",Date of Sample Collection,Station Latitude,Station Longitude,...,EDDs.MATRIX,EDDs.METHOD,EDDs.WORKGROUP,EDDs.SAMPLE_TYPE,EDDs.ANALYTE_TYPE,EDDs.ORDER,Project Qual (WPP),QC Comments (WPP),Comments,QC FB
0,20-0161,Merrimack River,2,1,Freshwater,W,G,2024-05-14,42.64954,-71.3845,...,AQUEOUS,SGS AXYS METHOD MLA-110 Rev 02,WG90240,SA,REG,10,-,NaN,NaN,NaN
1,20-0161,Merrimack River,2,1,Freshwater,W,G,2024-05-14,42.64954,-71.3845,...,AQUEOUS,SGS AXYS METHOD MLA-110 Rev 02,WG90240,SA,REG,20,j,Lab qualifier carried forward.,Lab qualifier carried forward. MDL < x < RDL,NaN
2,20-0161,Merrimack River,2,1,Freshwater,W,G,2024-05-14,42.64954,-71.3845,...,AQUEOUS,SGS AXYS METHOD MLA-110 Rev 02,WG90240,SA,REG,30,-,NaN,>RL,NaN
3,20-0161,Merrimack River,2,1,Freshwater,W,G,2024-05-14,42.64954,-71.3845,...,AQUEOUS,SGS AXYS METHOD MLA-110 Rev 02,WG90240,SA,REG,40,j,Lab qualifier carried forward.,Lab qualifier carried forward. MDL < x < RDL,NaN
4,20-0161,Merrimack River,2,1,Freshwater,W,G,2024-05-14,42.64954,-71.3845,...,AQUEOUS,SGS AXYS METHOD MLA-110 Rev 02,WG90240,SA,REG,50,-,NaN,>RL,MDL < x < RDL


(1600, 49)
['Project sampleID', 'Site Name', 'SiteID', 'Phase ', 'Freshwater or Coastal', 'Matrix', 'Sample Type (G,C,I)', 'Date of Sample Collection', 'Station Latitude', 'Station Longitude', 'Latitude', 'Longitude', 'Latitude 2', 'Longitude 2', 'Latitude 3', 'Longitude 3', 'QC sample', 'Sample Type', 'Lab ID', 'EDDs.SAMPLE_NO', 'EDDs.AXYS_ID', 'EDDs.ANALYSIS_DATE', 'EDDs.ANALYSIS', 'EDDs.COMPOUND', 'EDDs.CAS_NO', 'EDDs.IUPAC_NO', 'EDDs.COELUTIONS', 'EDDs.LAB_FLAG', 'EDDs.CONC_FOUND', 'EDDs.DETECTION_LIMIT', 'EDDs.UNIT', 'EDDs.ION_ABUNDANCE_RATIO', 'EDDs.RRT', 'EDDs.SAMPLE_SIZE', 'EDDs.SAMPLE_SIZE_UNIT', 'EDDs.EXTRACTION_DATE', 'EDDs.RECEIVED_DATE', 'EDDs.COLLECTION_DATE', 'EDDs.PROJECT_ID', 'EDDs.MATRIX', 'EDDs.METHOD', 'EDDs.WORKGROUP', 'EDDs.SAMPLE_TYPE', 'EDDs.ANALYTE_TYPE', 'EDDs.ORDER', 'Project Qual (WPP)', 'QC Comments (WPP)', 'Comments', 'QC FB']


In [59]:
# I keep and rename the MA water results columns so they match Phase I and Phase II.

ma_long_2024 = ma_metadata_2024[
    [
        "Project sampleID",
        "Date of Sample Collection",
        "Site Name",
        "Station Latitude",
        "Station Longitude",
        "EDDs.COMPOUND",
        "EDDs.CONC_FOUND",
        "EDDs.LAB_FLAG",
        "EDDs.DETECTION_LIMIT",
        "EDDs.UNIT"
    ]
].copy()

ma_long_2024 = ma_long_2024.rename(
    columns={
        "Project sampleID": "Sample ID",
        "Date of Sample Collection": "Sampling Date",
        "Site Name": "Waterbody",
        "Station Latitude": "Latitude",
        "Station Longitude": "Longitude",
        "EDDs.COMPOUND": "CharacteristicName",
        "EDDs.CONC_FOUND": "raw_result",
        "EDDs.LAB_FLAG": "MeasureQualifierCode",
        "EDDs.DETECTION_LIMIT": "DetectionQuantitationLimitMeasure/MeasureValue",
        "EDDs.UNIT": "ResultMeasure/MeasureUnitCode"
    }
)

ma_long_2024["Waterkeeper Group"] = "Massachusetts PFAS Monitoring Program"
ma_long_2024["State"] = "MA"
ma_long_2024["Upstream or Downstream"] = pd.NA

display(ma_long_2024.head())
print(ma_long_2024.shape)
print(ma_long_2024.columns.tolist())

,Sample ID,Sampling Date,Waterbody,Latitude,Longitude,CharacteristicName,raw_result,MeasureQualifierCode,DetectionQuantitationLimitMeasure/MeasureValue,ResultMeasure/MeasureUnitCode,Waterkeeper Group,State,Upstream or Downstream
0,20-0161,2024-05-14,Merrimack River,42.64954,-71.3845,PFBA,NaN,U,1.650,ng/L,Massachusetts PFAS Monitoring Program,MA,<NA>
1,20-0161,2024-05-14,Merrimack River,42.64954,-71.3845,PFPeA,2.03,J,0.823,ng/L,Massachusetts PFAS Monitoring Program,MA,<NA>
2,20-0161,2024-05-14,Merrimack River,42.64954,-71.3845,PFHxA,2.33,NaN,0.412,ng/L,Massachusetts PFAS Monitoring Program,MA,<NA>
3,20-0161,2024-05-14,Merrimack River,42.64954,-71.3845,PFHpA,1.26,J,0.412,ng/L,Massachusetts PFAS Monitoring Program,MA,<NA>
4,20-0161,2024-05-14,Merrimack River,42.64954,-71.3845,PFOA,3.53,NaN,0.412,ng/L,Massachusetts PFAS Monitoring Program,MA,<NA>


(1600, 13)
['Sample ID', 'Sampling Date', 'Waterbody', 'Latitude', 'Longitude', 'CharacteristicName', 'raw_result', 'MeasureQualifierCode', 'DetectionQuantitationLimitMeasure/MeasureValue', 'ResultMeasure/MeasureUnitCode', 'Waterkeeper Group', 'State', 'Upstream or Downstream']


In [60]:
# I clean the MA result values.

ma_long_2024["raw_result_text"] = (
    ma_long_2024["raw_result"]
    .astype("string")
    .str.strip()
)

ma_long_2024["ResultMeasureValue_num"] = pd.to_numeric(
    ma_long_2024["raw_result_text"],
    errors="coerce"
)

display(ma_long_2024[["raw_result_text", "ResultMeasureValue_num", "MeasureQualifierCode"]].head(20))

,raw_result_text,ResultMeasureValue_num,MeasureQualifierCode
0,<NA>,<NA>,U
1,2.03,2.03,J
2,2.33,2.33,NaN
3,1.26,1.26,J
4,3.53,3.53,NaN
5,<NA>,<NA>,U
6,<NA>,<NA>,U
7,<NA>,<NA>,U
8,<NA>,<NA>,U
9,<NA>,<NA>,U


In [61]:
# I review all qualifier codes in the MA dataset.

display(
    ma_long_2024["MeasureQualifierCode"]
    .value_counts(dropna=False)
)

MeasureQualifierCode
U      1479
J        74
NaN      43
K J       3
K         1
Name: count, dtype: int64

In [62]:
display(
    ma_long_2024.loc[
        ma_long_2024["MeasureQualifierCode"].isin(["K", "K J"]),
        [
            "CharacteristicName",
            "raw_result_text",
            "ResultMeasureValue_num",
            "MeasureQualifierCode"
        ]
    ]
)

,CharacteristicName,raw_result_text,ResultMeasureValue_num,MeasureQualifierCode
245,PFNA,0.446,0.446,K J
346,EtFOSAA,0.467,0.467,K J
540,6:2 FTS,18.6,18.6,K
924,PFOA,0.488,0.488,K J


In [63]:
# Tony ( I havent changed this based on index tab)
# based on this file:https://www.millisma.gov/sites/g/files/vyhlif901/f/uploads/pfas_testing_results_collected_october_21_2020.pdf
# I think 
# J means estimated value so Y
# for U I found this for Washington state:https://apps.ecology.wa.gov/webdmrview/HelpPages/ValueQualifierHelp.aspx
# for K I found this for Florida:https://floridadep.gov/sites/default/files/62-160_help-document_0.pdf
# MA:https://www.mass.gov/doc/pfas-qc-guidance-data-report-content-for-edep/download

In [64]:
# I clean the MA qualifier code before assigning PFAS detection.

qualifier_clean = (
    ma_long_2024["MeasureQualifierCode"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.strip()
)

# I create the PFAS detection flag using the MA qualifier definitions.

ma_long_2024["pfas_detected"] = np.select(
    [
        qualifier_clean.eq("U"),
        qualifier_clean.eq("J"),
        qualifier_clean.eq("K"),
        qualifier_clean.eq("K J")
    ],
    [
        "N",
        "Y",
        "Y",
        "Y"
    ],
    default=pd.NA
)

display(
    ma_long_2024["pfas_detected"]
    .value_counts(dropna=False)
)

pfas_detected
N       1479
Y         78
<NA>      43
Name: count, dtype: int64

In [65]:
# I review rows where PFAS detection is still missing.

na_pfas = ma_long_2024.loc[
    ma_long_2024["pfas_detected"].isna(),
    [
        "Sample ID",
        "Waterbody",
        "CharacteristicName",
        "raw_result",
        "raw_result_text",
        "MeasureQualifierCode",
        "DetectionQuantitationLimitMeasure/MeasureValue"
    ]
]

display(na_pfas)
print("Rows:", len(na_pfas))

,Sample ID,Waterbody,CharacteristicName,raw_result,raw_result_text,MeasureQualifierCode,DetectionQuantitationLimitMeasure/MeasureValue
2,20-0161,Merrimack River,PFHxA,2.33,2.33,NaN,0.412
4,20-0161,Merrimack River,PFOA,3.53,3.53,NaN,0.412
15,20-0161,Merrimack River,PFOS,2.10,2.1,NaN,0.412
281,20-0173,Charles River,PFPeA,3.69,3.69,NaN,0.843
282,20-0173,Charles River,PFHxA,3.88,3.88,NaN,0.421
283,20-0173,Charles River,PFHpA,1.84,1.84,NaN,0.421
284,20-0173,Charles River,PFOA,5.33,5.33,NaN,0.421
291,20-0173,Charles River,PFBS,2.64,2.64,NaN,0.421
293,20-0173,Charles River,PFHxS,2.32,2.32,NaN,0.484
295,20-0173,Charles River,PFOS,7.26,7.26,NaN,0.421


Rows: 43


In [66]:
# I classify rows with a reported concentration and no qualifier as detections.

mask_detected_no_qualifier = (
    ma_long_2024["pfas_detected"].isna()
    & ma_long_2024["ResultMeasureValue_num"].notna()
)

ma_long_2024.loc[
    mask_detected_no_qualifier,
    "pfas_detected"
] = "Y"

display(ma_long_2024["pfas_detected"].value_counts(dropna=False))

pfas_detected
N    1479
Y     121
Name: count, dtype: int64

In [67]:
# I reorder the MA file to match Phase I and Phase II.

ma_long_2024 = ma_long_2024[
    [
        "Sample ID",
        "Sampling Date",
        "Waterkeeper Group",
        "State",
        "Waterbody",
        "Upstream or Downstream",
        "Latitude",
        "Longitude",
        "CharacteristicName",
        "raw_result",
        "raw_result_text",
        "pfas_detected",
        "MeasureQualifierCode",
        "DetectionQuantitationLimitMeasure/MeasureValue"
    ]
].copy()

print(ma_long_2024.shape)
print(ma_long_2024.columns.tolist())
display(ma_long_2024.head())

(1600, 14)
['Sample ID', 'Sampling Date', 'Waterkeeper Group', 'State', 'Waterbody', 'Upstream or Downstream', 'Latitude', 'Longitude', 'CharacteristicName', 'raw_result', 'raw_result_text', 'pfas_detected', 'MeasureQualifierCode', 'DetectionQuantitationLimitMeasure/MeasureValue']


,Sample ID,Sampling Date,Waterkeeper Group,State,Waterbody,Upstream or Downstream,Latitude,Longitude,CharacteristicName,raw_result,raw_result_text,pfas_detected,MeasureQualifierCode,DetectionQuantitationLimitMeasure/MeasureValue
0,20-0161,2024-05-14,Massachusetts PFAS Monitoring Program,MA,Merrimack River,<NA>,42.64954,-71.3845,PFBA,NaN,<NA>,N,U,1.650
1,20-0161,2024-05-14,Massachusetts PFAS Monitoring Program,MA,Merrimack River,<NA>,42.64954,-71.3845,PFPeA,2.03,2.03,Y,J,0.823
2,20-0161,2024-05-14,Massachusetts PFAS Monitoring Program,MA,Merrimack River,<NA>,42.64954,-71.3845,PFHxA,2.33,2.33,Y,NaN,0.412
3,20-0161,2024-05-14,Massachusetts PFAS Monitoring Program,MA,Merrimack River,<NA>,42.64954,-71.3845,PFHpA,1.26,1.26,Y,J,0.412
4,20-0161,2024-05-14,Massachusetts PFAS Monitoring Program,MA,Merrimack River,<NA>,42.64954,-71.3845,PFOA,3.53,3.53,Y,NaN,0.412


### MA 2022

In [68]:
## Tony this is NEW

In [69]:
# I load the Massachusetts 2022 PFAS water data.
# I use W_Data because this project is about water results, not fish tissue results.

ma_2022_path = file_path + "raw_data/MA data/DEP PFAS in water Database 09_12_23_final.xlsx"

ma_2022_raw = pd.read_excel(
    ma_2022_path,
    sheet_name="W_Data"
)

display(ma_2022_raw.head())
print(ma_2022_raw.shape)
print(ma_2022_raw.columns.tolist())

,LabID,LabSNum,FieldSampNum,Analyte/Characteristic,Analyte/Abbrv,Sample Fraction,Result,LabQual,ResComm,Units,...,Result_Num,PFAS4_Flag,Waterbody,SiteID,Sampling Location Long,Sampling Location Lat,Reference,QC sample (Y/N),"Sample Type (R, B, D, FB, EB)",Primary sampleID (for Dups)
0,Eurofins Lancaster Laboratories Environment Te...,410-86865-2,20-0008,11Cl-PF3OUdS,11Cl-PF3OUdS,Total,<MDL,U,NaN,ng/L,...,NaN,NaN,Ashumet Pond,5,-70.533653,41.631657,NaN,N,R,--
1,Eurofins Lancaster Laboratories Environment Te...,410-86865-2,20-0008,3:3 FTCA,3:3FTCA,Total,<MDL,U,NaN,ng/L,...,NaN,NaN,Ashumet Pond,5,-70.533653,41.631657,NaN,N,R,--
2,Eurofins Lancaster Laboratories Environment Te...,410-86865-2,20-0008,4:2 FTS,4:2FTS,Total,<MDL,U,NaN,ng/L,...,NaN,NaN,Ashumet Pond,5,-70.533653,41.631657,NaN,N,R,--
3,Eurofins Lancaster Laboratories Environment Te...,410-86865-2,20-0008,5:3 FTCA,5:3FTCA,Total,<MDL,U,NaN,ng/L,...,NaN,NaN,Ashumet Pond,5,-70.533653,41.631657,NaN,N,R,--
4,Eurofins Lancaster Laboratories Environment Te...,410-86865-2,20-0008,6:2 FTS,6:2FTS,Total,3.5,J,carried lab qual forward and translated to WPP...,ng/L,...,3.5,NaN,Ashumet Pond,5,-70.533653,41.631657,NaN,N,R,--


(2640, 31)
['LabID', 'LabSNum', 'FieldSampNum', 'Analyte/Characteristic', 'Analyte/Abbrv', 'Sample Fraction', 'Result', 'LabQual', 'ResComm', 'Units', 'MDL', 'RDL', 'UQL', 'Matrix', 'Analytical Method', 'AnalDate', 'AnalTime', 'SiteLocator', 'CollectDate', 'CollectTime', 'ProjectQual', 'Result_Num', 'PFAS4_Flag', 'Waterbody', 'SiteID', 'Sampling Location Long', 'Sampling Location Lat', 'Reference', 'QC sample (Y/N)', 'Sample Type (R, B, D, FB, EB)', 'Primary sampleID (for Dups)']


In [70]:
# I keep and rename the MA 2022 water columns so they match the other separate datasets.

ma_2022_long = ma_2022_raw[
    [
        "FieldSampNum",
        "CollectDate",
        "Waterbody",
        "Sampling Location Lat",
        "Sampling Location Long",
        "Analyte/Characteristic",
        "Result",
        "LabQual",
        "MDL",
        "RDL",
        "Units",
        "Matrix",
        "QC sample (Y/N)",
        "Sample Type (R, B, D, FB, EB)"
    ]
].copy()

ma_2022_long = ma_2022_long.rename(
    columns={
        "FieldSampNum": "Sample ID",
        "CollectDate": "Sampling Date",
        "Waterbody": "Waterbody",
        "Sampling Location Lat": "Latitude",
        "Sampling Location Long": "Longitude",
        "Analyte/Characteristic": "CharacteristicName",
        "Result": "raw_result",
        "LabQual": "MeasureQualifierCode",
        "MDL": "DetectionQuantitationLimitMeasure/MeasureValue",
        "RDL": "ReportingDetectionLimit",
        "Units": "ResultMeasure/MeasureUnitCode"
    }
)

ma_2022_long["Waterkeeper Group"] = "Massachusetts DEP PFAS Monitoring Program"
ma_2022_long["State"] = "MA"
ma_2022_long["Upstream or Downstream"] = pd.NA

display(ma_2022_long.head())
print(ma_2022_long.shape)

,Sample ID,Sampling Date,Waterbody,Latitude,Longitude,CharacteristicName,raw_result,MeasureQualifierCode,DetectionQuantitationLimitMeasure/MeasureValue,ReportingDetectionLimit,ResultMeasure/MeasureUnitCode,Matrix,QC sample (Y/N),"Sample Type (R, B, D, FB, EB)",Waterkeeper Group,State,Upstream or Downstream
0,20-0008,06/06/2022,Ashumet Pond,41.631657,-70.533653,11Cl-PF3OUdS,<MDL,U,2.2,8.2,ng/L,Water,N,R,Massachusetts DEP PFAS Monitoring Program,MA,<NA>
1,20-0008,06/06/2022,Ashumet Pond,41.631657,-70.533653,3:3 FTCA,<MDL,U,1.5,10.0,ng/L,Water,N,R,Massachusetts DEP PFAS Monitoring Program,MA,<NA>
2,20-0008,06/06/2022,Ashumet Pond,41.631657,-70.533653,4:2 FTS,<MDL,U,1.8,8.2,ng/L,Water,N,R,Massachusetts DEP PFAS Monitoring Program,MA,<NA>
3,20-0008,06/06/2022,Ashumet Pond,41.631657,-70.533653,5:3 FTCA,<MDL,U,10.0,52.0,ng/L,Water,N,R,Massachusetts DEP PFAS Monitoring Program,MA,<NA>
4,20-0008,06/06/2022,Ashumet Pond,41.631657,-70.533653,6:2 FTS,3.5,J,2.6,8.2,ng/L,Water,N,R,Massachusetts DEP PFAS Monitoring Program,MA,<NA>


(2640, 17)


In [71]:
# I clean the MA 2022 result values and dates.

ma_2022_long["raw_result_text"] = (
    ma_2022_long["raw_result"]
    .astype("string")
    .str.strip()
)

ma_2022_long["ResultMeasureValue_num"] = pd.to_numeric(
    ma_2022_long["raw_result_text"],
    errors="coerce"
)

ma_2022_long["Sampling Date"] = pd.to_datetime(
    ma_2022_long["Sampling Date"],
    errors="coerce"
)

display(
    ma_2022_long[
        [
            "Sample ID",
            "Sampling Date",
            "Waterbody",
            "CharacteristicName",
            "raw_result_text",
            "ResultMeasureValue_num",
            "MeasureQualifierCode"
        ]
    ].head(20)
)

,Sample ID,Sampling Date,Waterbody,CharacteristicName,raw_result_text,ResultMeasureValue_num,MeasureQualifierCode
0,20-0008,2022-06-06,Ashumet Pond,11Cl-PF3OUdS,<MDL,<NA>,U
1,20-0008,2022-06-06,Ashumet Pond,3:3 FTCA,<MDL,<NA>,U
2,20-0008,2022-06-06,Ashumet Pond,4:2 FTS,<MDL,<NA>,U
3,20-0008,2022-06-06,Ashumet Pond,5:3 FTCA,<MDL,<NA>,U
4,20-0008,2022-06-06,Ashumet Pond,6:2 FTS,3.5,3.5,J
5,20-0008,2022-06-06,Ashumet Pond,7:3 FTCA,<MDL,<NA>,U
6,20-0008,2022-06-06,Ashumet Pond,8:2 FTS,<MDL,<NA>,U
7,20-0008,2022-06-06,Ashumet Pond,9Cl-PF3ONS,<MDL,<NA>,U
8,20-0008,2022-06-06,Ashumet Pond,DONA,<MDL,<NA>,U
9,20-0008,2022-06-06,Ashumet Pond,HFPO-DA,<MDL,<NA>,U


In [72]:
# I review the MA 2022 qualifier codes before assigning PFAS detection.

display(
    ma_2022_long["MeasureQualifierCode"]
    .value_counts(dropna=False)
)

display(
    ma_2022_long["raw_result_text"]
    .value_counts(dropna=False)
    .head(30)
)

display(
    ma_2022_long["ResultMeasure/MeasureUnitCode"]
    .value_counts(dropna=False)
)

display(
    ma_2022_long["Matrix"]
    .value_counts(dropna=False)
)

MeasureQualifierCode
U          2115
NaN         278
J           210
J I          13
U cn         10
I             8
U *+          4
cn            1
B *+ cn       1
Name: count, dtype: int64

raw_result_text
<MDL    2128
2.6       11
1.3       11
4.2       11
2.3       11
1.9       10
3.4       10
3.9       10
4.1       10
1.1       10
3.8       10
2.0        9
1.2        9
1.6        9
3.7        9
1.0        8
4.4        8
2.9        8
7.0        8
1.5        8
3.6        7
1.8        7
2.2        7
2.1        7
2.4        6
13         6
1.4        6
10         6
2.8        6
3.0        6
Name: count, dtype: Int64

ResultMeasure/MeasureUnitCode
ng/L    2640
Name: count, dtype: int64

Matrix
Water    2640
Name: count, dtype: int64

In [73]:
# I review all <MDL results and their qualifier codes.

display(
    ma_2022_long.loc[
        ma_2022_long["raw_result_text"].str.upper().eq("<MDL"),
        [
            "CharacteristicName",
            "raw_result_text",
            "MeasureQualifierCode",
            "DetectionQuantitationLimitMeasure/MeasureValue",
            "ResultMeasureValue_num"
        ]
    ]
)

print(
    ma_2022_long.loc[
        ma_2022_long["raw_result_text"].str.upper().eq("<MDL"),
        "MeasureQualifierCode"
    ].value_counts(dropna=False)
)

,CharacteristicName,raw_result_text,MeasureQualifierCode,DetectionQuantitationLimitMeasure/MeasureValue,ResultMeasureValue_num
0,11Cl-PF3OUdS,<MDL,U,2.2,<NA>
1,3:3 FTCA,<MDL,U,1.5,<NA>
2,4:2 FTS,<MDL,U,1.8,<NA>
3,5:3 FTCA,<MDL,U,10.0,<NA>
5,7:3 FTCA,<MDL,U,10.0,<NA>
...,...,...,...,...,...
2635,Perfluoroundecanoic acid,<MDL,U,0.5,<NA>
2636,PES,<MDL,U,0.5,<NA>
2637,PFECA A,<MDL,U,1.0,<NA>
2638,PFECA B,<MDL,U,1.0,<NA>


MeasureQualifierCode
U       2114
U cn      10
U *+       4
Name: count, dtype: int64


data dicionary tab
*+	LCS and/or LCSD is outside acceptance limits, high biased.
*5+	Isotope dilution analyte is outside acceptance limits, high biased.
cn	Refer to Case Narrative for further detail
H	Sample was prepped or analyzed beyond the specified holding time
I	Value is EMPC (estimated maximum possible concentration).
J	Result is less than the RL but greater than or equal to the MDL and the concentration is an approximate value.
U	Non_Detected
E	Result exceeded calibration range.
B	Compound was found in the blank and sample.
*5-	Isotope dilution analyte is outside acceptance limits, low biased.

In [74]:
# I remove records with the U *+ qualifier.

print("Rows before:", len(ma_2022_long))

ma_2022_long = ma_2022_long.loc[
    ma_2022_long["MeasureQualifierCode"] != "U *+"
].copy()

print("Rows after:", len(ma_2022_long))

Rows before: 2640
Rows after: 2636


In [75]:
# I assign the PFAS detection flag for MA 2022.
# I treat U and <MDL as not detected.
# I treat any numeric result as detected, including estimated J results.

ma_2022_long["qualifier_clean"] = (
    ma_2022_long["MeasureQualifierCode"]
    .astype("string")
    .str.upper()
    .str.strip()
)

ma_2022_long["pfas_detected"] = np.select(
    [
        ma_2022_long["ResultMeasureValue_num"].notna(),
        ma_2022_long["qualifier_clean"].str.contains("U", na=False),
        ma_2022_long["raw_result_text"].str.upper().eq("<MDL")
    ],
    [
        "Y",
        "N",
        "N"
    ],
    default=pd.NA
)

display(ma_2022_long["pfas_detected"].value_counts(dropna=False))

pfas_detected
N    2125
Y     511
Name: count, dtype: int64

In [76]:
# I check any MA 2022 rows that are still not classified.

ma_2022_unclassified = ma_2022_long.loc[
    ma_2022_long["pfas_detected"].isna(),
    [
        "Sample ID",
        "Waterbody",
        "CharacteristicName",
        "raw_result",
        "raw_result_text",
        "ResultMeasureValue_num",
        "MeasureQualifierCode",
        "DetectionQuantitationLimitMeasure/MeasureValue"
    ]
]

display(ma_2022_unclassified)
print("Unclassified rows:", len(ma_2022_unclassified))

,Sample ID,Waterbody,CharacteristicName,raw_result,raw_result_text,ResultMeasureValue_num,MeasureQualifierCode,DetectionQuantitationLimitMeasure/MeasureValue


Unclassified rows: 0


In [77]:
# I keep the final MA 2022 columns in the same structure as the other separate datasets.

ma_2022_long = ma_2022_long[
    [
        "Sample ID",
        "Sampling Date",
        "Waterkeeper Group",
        "State",
        "Waterbody",
        "Upstream or Downstream",
        "Latitude",
        "Longitude",
        "CharacteristicName",
        "raw_result",
        "raw_result_text",
        "pfas_detected",
        "MeasureQualifierCode",
        "DetectionQuantitationLimitMeasure/MeasureValue"
    ]
].copy()

print(ma_2022_long.shape)
print(ma_2022_long.columns.tolist())
display(ma_2022_long.head())

(2636, 14)
['Sample ID', 'Sampling Date', 'Waterkeeper Group', 'State', 'Waterbody', 'Upstream or Downstream', 'Latitude', 'Longitude', 'CharacteristicName', 'raw_result', 'raw_result_text', 'pfas_detected', 'MeasureQualifierCode', 'DetectionQuantitationLimitMeasure/MeasureValue']


,Sample ID,Sampling Date,Waterkeeper Group,State,Waterbody,Upstream or Downstream,Latitude,Longitude,CharacteristicName,raw_result,raw_result_text,pfas_detected,MeasureQualifierCode,DetectionQuantitationLimitMeasure/MeasureValue
0,20-0008,2022-06-06,Massachusetts DEP PFAS Monitoring Program,MA,Ashumet Pond,<NA>,41.631657,-70.533653,11Cl-PF3OUdS,<MDL,<MDL,N,U,2.2
1,20-0008,2022-06-06,Massachusetts DEP PFAS Monitoring Program,MA,Ashumet Pond,<NA>,41.631657,-70.533653,3:3 FTCA,<MDL,<MDL,N,U,1.5
2,20-0008,2022-06-06,Massachusetts DEP PFAS Monitoring Program,MA,Ashumet Pond,<NA>,41.631657,-70.533653,4:2 FTS,<MDL,<MDL,N,U,1.8
3,20-0008,2022-06-06,Massachusetts DEP PFAS Monitoring Program,MA,Ashumet Pond,<NA>,41.631657,-70.533653,5:3 FTCA,<MDL,<MDL,N,U,10.0
4,20-0008,2022-06-06,Massachusetts DEP PFAS Monitoring Program,MA,Ashumet Pond,<NA>,41.631657,-70.533653,6:2 FTS,3.5,3.5,Y,J,2.6


In [78]:
# I save the cleaned Massachusetts 2022 PFAS water results.

output_path = file_path + "saved files/ma_2022_water_long.csv"

ma_2022_long.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows:", len(ma_2022_long))

Saved: G:/My Drive/Projects/Large projects/PFAS in water/saved files/ma_2022_water_long.csv
Rows: 2636


In [79]:
# I save the cleaned Massachusetts PFAS water results.

output_path = file_path + "saved files/ma_2024_water_long.csv"

ma_long_2024.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)

Saved: G:/My Drive/Projects/Large projects/PFAS in water/saved files/ma_2024_water_long.csv


In [80]:
# I combine the Massachusetts 2022 and 2024 PFAS water datasets.

ma_long = pd.concat(
    [
        ma_2022_long,
        ma_long_2024
    ],
    ignore_index=True
)

print("Rows in MA 2022:", len(ma_2022_long))
print("Rows in MA 2024:", len(ma_long_2024))
print("Rows in MA after merge:", len(ma_long))

display(ma_long.head())

Rows in MA 2022: 2636
Rows in MA 2024: 1600
Rows in MA after merge: 4236


,Sample ID,Sampling Date,Waterkeeper Group,State,Waterbody,Upstream or Downstream,Latitude,Longitude,CharacteristicName,raw_result,raw_result_text,pfas_detected,MeasureQualifierCode,DetectionQuantitationLimitMeasure/MeasureValue
0,20-0008,2022-06-06,Massachusetts DEP PFAS Monitoring Program,MA,Ashumet Pond,<NA>,41.631657,-70.533653,11Cl-PF3OUdS,<MDL,<MDL,N,U,2.2
1,20-0008,2022-06-06,Massachusetts DEP PFAS Monitoring Program,MA,Ashumet Pond,<NA>,41.631657,-70.533653,3:3 FTCA,<MDL,<MDL,N,U,1.5
2,20-0008,2022-06-06,Massachusetts DEP PFAS Monitoring Program,MA,Ashumet Pond,<NA>,41.631657,-70.533653,4:2 FTS,<MDL,<MDL,N,U,1.8
3,20-0008,2022-06-06,Massachusetts DEP PFAS Monitoring Program,MA,Ashumet Pond,<NA>,41.631657,-70.533653,5:3 FTCA,<MDL,<MDL,N,U,10.0
4,20-0008,2022-06-06,Massachusetts DEP PFAS Monitoring Program,MA,Ashumet Pond,<NA>,41.631657,-70.533653,6:2 FTS,3.5,3.5,Y,J,2.6


In [81]:
# I verify that both years are present.

ma_long["Sampling Date"] = pd.to_datetime(
    ma_long["Sampling Date"],
    errors="coerce"
)

display(
    ma_long["Sampling Date"]
    .dt.year
    .value_counts()
    .sort_index()
)

display(
    ma_long["pfas_detected"]
    .value_counts(dropna=False)
)

Sampling Date
2022    2636
2024    1600
Name: count, dtype: int64

pfas_detected
N    3604
Y     632
Name: count, dtype: int64

In [82]:
# I save the combined Massachusetts PFAS water dataset.

output_path = file_path + "saved files/ma_long.csv"

ma_long.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)

Saved: G:/My Drive/Projects/Large projects/PFAS in water/saved files/ma_long.csv
